<a href="https://colab.research.google.com/github/duongntsusu/A-Study-of-Cinematic-Neural-Style-Transfer/blob/main/First_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# force_remount = True

Mounted at /content/drive


In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_pil_image
from torch.utils.data import Dataset
import torchvision.transforms as T
from PIL import Image
from torchvision import datasets
from torch.utils.data import DataLoader
from pathlib import Path

# import for main
import random
import torch
import argparse
from pathlib import Path
from torch.optim import Adam
from torchvision.models import vgg16, VGG16_Weights
from torchvision.models.feature_extraction import create_feature_extractor

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
normalize = T.Normalize(mean=MEAN, std=STD)
denormalize = T.Normalize(mean=[-m/s for m, s in zip(MEAN, STD)],
                          std=[1/std for std in STD])

def get_transforms(imsize=None, cropsize=None, cencrop=False):
    """Get the transforms."""
    transformer = []
    if imsize:
        transformer.append(T.Resize(imsize))
    if cropsize:
        if cencrop:
            transformer.append(T.CenterCrop(cropsize))
        else:
            transformer.append(T.RandomCrop(cropsize))

    transformer.append(T.ToTensor())
    transformer.append(normalize)
    return T.Compose(transformer)


def imload(path, imsize=None, cropsize=None, cencrop=False):
    """Load a image."""
    transformer = get_transforms(imsize=imsize,
                                 cropsize=cropsize,
                                 cencrop=cencrop)
    image = Image.open(path).convert("RGB")
    return transformer(image).unsqueeze(0)


def imsave(image, save_path):
    """Save a image."""
    image = denormalize(torchvision.utils.make_grid(image)).clamp_(0.0, 1.0)
    torchvision.utils.save_image(image, save_path)
    return None

class ImageDataset:
    """Image Dataset."""

    def __init__(self, dir_path):
        """Init."""
        self.images = sorted(list(dir_path.glob('*.jpg'))+list(dir_path.glob('*.png')))

    def __len__(self):
        """Return the Number of data sampels."""
        return len(self.images)

    def __getitem__(self, index):
        """Get Image and Index."""
        img = Image.open(self.images[index]).convert('RGB')
        return img, index


class StyleDataset:
    def __init__(self, dir_path):
        self.images = []
        style_subdirs = sorted([d for d in dir_path.iterdir() if d.is_dir() and not d.name.startswith('.')])

        for style_id, style_subdir in enumerate(style_subdirs):
            self.images.extend([(img_path, style_id) for img_path in style_subdir.glob('*.png')]+[(img_path, style_id) for img_path in style_subdir.glob('*.jpg')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path, style_id = self.images[index]
        img = Image.open(img_path).convert('RGB')
        return img, style_id


class DataProcessor:
  def __init__(self, imsize = 256, cropsize = 240, cencrop = False):
    self. transforms = get_transforms(imsize = imsize, cropsize = cropsize, cencrop = cencrop)
  def __call__(self, batch):
        """Process the batch."""
        images, indices = list(zip(*batch))

        inputs = torch.stack([self.transforms(image) for image in images])
        return inputs, indices

def resize(np_img,basewidth=200):
    '''resize the image'''
    img = Image.fromarray(np_img)
    wpercent = (basewidth / float(img.size[0]))
    hsize = int((float(img.size[1]) * float(wpercent)))
    img = img.resize((basewidth, hsize), Image.LANCZOS)
    return np.array(img)

In [ ]:
# Step 1:
style_dataset_path = "/content/drive/MyDrive/Movie_Project/Directors_still_images"
content_dataset_path = "/content/drive/MyDrive/Movie_Project/Content_image/"
testing_dataset_path = "/content/drive/MyDrive/Movie_Project/Testing_dataset"

style_dataset = StyleDataset(dir_path = Path(style_dataset_path))
content_dataset = ImageDataset(dir_path = Path(content_dataset_path))
testing_dataset = ImageDataset(dir_path = Path(testing_dataset_path))
data_processor = DataProcessor(imsize = 256, cropsize = 240, cencrop = False)

# Step 2:
style_dataloader = torch.utils.data.DataLoader(style_dataset, batch_size = 8, shuffle = True, collate_fn = data_processor)
content_dataloader = torch.utils.data.DataLoader(content_dataset, batch_size = 8, shuffle = True, collate_fn = data_processor)
testing_data_processor = DataProcessor(imsize=256, cropsize=240, cencrop=True)
testing_dataloader = torch.utils.data.DataLoader(
    testing_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=testing_data_processor
)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_STYLE = 4

In [ ]:
class StyleTransferNetwork(nn.Module):
    """Style Transfer Network."""

    def __init__(self, num_style=4):
        """Init."""
        super(StyleTransferNetwork, self).__init__()
        self.conv1 = ConvWithCIN(num_style,  3, 32, 1, 'relu', 9)
        self.conv2 = ConvWithCIN(num_style, 32, 64, 2, 'relu', 3)
        self.conv3 = ConvWithCIN(num_style, 64, 128, 2, 'relu', 3)

        self.residual1 = ResidualBlock(num_style, 128, 128)
        self.residual2 = ResidualBlock(num_style, 128, 128)
        self.residual3 = ResidualBlock(num_style, 128, 128)
        self.residual4 = ResidualBlock(num_style, 128, 128)
        self.residual5 = ResidualBlock(num_style, 128, 128)

        self.upsampling1 = UpsamleBlock(num_style, 128, 64)
        self.upsampling2 = UpsamleBlock(num_style, 64, 32)

        self.conv4 = ConvWithCIN(num_style, 32, 3, 1, 'linear', 9)

        """
        init_style_weight = torch.tensor(10.0)
        init_content_weight = torch.tensor(1.0)
        init_tv_weight = torch.tensor(1e-5)

        self.log_style_weight = nn.Parameter(torch.log(init_style_weight))
        self.log_content_weight = nn.Parameter(torch.log(init_content_weight))
        self.log_tv_weight = nn.Parameter(torch.log(init_tv_weight))

        The lines above are implementing style weight and content weight as part of the model.
        That is, they are learnable parameters that can be updated via calling the Adam optimizer
        in the train function. But that shouldn't be it.
        """



    def forward(self, x, style_codes):
        x = self.conv1(x, style_codes)
        x = self.conv2(x, style_codes)
        x = self.conv3(x, style_codes)

        x = self.residual1(x, style_codes)
        x = self.residual2(x, style_codes)
        x = self.residual3(x, style_codes)
        x = self.residual4(x, style_codes)
        x = self.residual5(x, style_codes)

        x = self.upsampling1(x, style_codes)
        x = self.upsampling2(x, style_codes)

        x = self.conv4(x, style_codes)

        return x

class CIN(nn.Module):
    """Conditional Instance Normalization"""

    def __init__(self, num_style, ch):
        super(CIN, self).__init__()
        self.normalize = nn.InstanceNorm2d(ch, affine=False)
        self.offset = nn.Parameter(0.01 * torch.randn(1, num_style, ch))
        self.scale = nn.Parameter(1 + 0.01 * torch.randn(1, num_style, ch))

    def forward(self, x, style_codes):
        b, c, h, w = x.size()
        x = self.normalize(x)
        gamma = torch.sum(self.scale * style_codes, dim=1).view(b, c, 1, 1)
        beta = torch.sum(self.offset * style_codes, dim=1).view(b, c, 1, 1)
        x = x * gamma + beta

        return x.view(b, c, h, w)


class ConvWithCIN(nn.Module):
    """Convolution layer with CIN."""

    def __init__(self, num_style, in_ch, out_ch, stride, activation, ksize):
        super(ConvWithCIN, self).__init__()
        self.padding = nn.ReflectionPad2d(ksize // 2)
        self.conv = nn.Conv2d(in_ch, out_ch, ksize, stride)

        self.cin = CIN(num_style, out_ch)
        if activation == "relu":
            self.activation = nn.ReLU()

        elif activation == "linear":
            self.activation = lambda x: x

    def forward(self, x, style_codes):
        x = self.padding(x)
        x = self.conv(x)
        x = self.cin(x, style_codes)
        x = self.activation(x)

        return x


class ResidualBlock(nn.Module):

    def __init__(self, num_style, in_ch, out_ch):
        super(ResidualBlock, self).__init__()
        self.conv1 = ConvWithCIN(num_style, in_ch, out_ch, 1, "relu", 3)
        self.conv2 = ConvWithCIN(num_style, out_ch, out_ch, 1, "linear", 3)

    def forward(self, x, style_codes):
        out = self.conv1(x, style_codes)
        out = self.conv2(out, style_codes)

        return x + out


class UpsamleBlock(nn.Module):
    def __init__(self, num_style, in_ch, out_ch):
        super(UpsamleBlock, self).__init__()
        self.conv = ConvWithCIN(num_style, in_ch, out_ch, 1, "relu", 3)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, x, style_codes):
        x = self.upsample(x)
        x = self.conv(x, style_codes)

        return x


In [ ]:
#loss
import torch
from torch.nn.functional import mse_loss


def calc_content_loss(features, targets, nodes):
    content_loss = 0
    for node in nodes:
        content_loss += mse_loss(features[node], targets[node])
    return content_loss


def gram(x):
    b, c, h, w = x.size()
    f = x.flatten(2)
    g = torch.bmm(f, f.transpose(1, 2))
    return g.div(h*w)


def calc_style_loss(features, targets, nodes):
    gram_loss = 0
    for node in nodes:
        gram_loss += mse_loss(gram(features[node]), gram(targets[node]))
    return gram_loss

def calc_style_loss_custom(features, targets, nodes):
    gram_loss = 0
    for node in nodes:
        # Calculate gram matrix of targets[node] (which is a 4D feature map) to get a 3D gram matrix.
        gram_target = gram(targets[node]) # shape will be [1, C, C]
        # Calculate gram matrix of features[node] (which is a 4D feature map) to get a 3D gram matrix.
        gram_output = gram(features[node]) # shape will be [B, C, C]
        expanded_gram_target = gram_target.expand_as(gram_output)

        gram_loss += mse_loss(gram_output, expanded_gram_target)
    return gram_loss

def calc_tv_loss(x): # Total variation loss helps to reduce the noise in the stylized image
    tv_loss = torch.mean(torch.abs(x[:, :, :, :-1] - x[:, :, :, 1:]))
    tv_loss += torch.mean(torch.abs(x[:, :, :-1, :] - x[:, :, 1:, :]))
    return tv_loss

In [ ]:

import copy
import math
import time
import random
import numpy as np
import pandas as pd
import torch
from scipy import stats


SEEDS = [11, 22, 33]
DEFAULT_NUM_EPOCHS = 300
CONVERGENCE_TOLERANCE = 0.05
CONVERGENCE_PATIENCE = 3
VALIDATION_INTERVAL = 10
EARLY_STOPPING_PATIENCE = 5
MIN_RELATIVE_IMPROVEMENT = 0.005

# Experiment 1: one representative image for each cinematic style.
# These are the exact paths from the original notebook.
STYLE_IMAGE_PATHS = {
    0: "/content/drive/MyDrive/Movie_Project/Directors_still_images/Tim_Alice/aliceinwonderland34.jpg",
    1: "/content/drive/MyDrive/Movie_Project/Directors_still_images/Tim_Beetlejuice/19 (130).jpg",
    2: "/content/drive/MyDrive/Movie_Project/Directors_still_images/We_Budapest/Budapest20.jpg",
    3: "/content/drive/MyDrive/Movie_Project/Directors_still_images/We_Moonrise/Moonrise2.png",
}


def set_reproducible_seed(seed):
    """Reset all random generators before every complete training run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cuda_sync():
    """Make elapsed-time measurements accurate when CUDA is used."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def process_single_image(image_path, imsize, cropsize, device):
    """Use a deterministic center crop for every style target."""
    img_pil = Image.open(image_path).convert("RGB")
    transform_func = get_transforms(
        imsize=imsize, cropsize=cropsize, cencrop=True
    )
    return transform_func(img_pil).unsqueeze(0).to(device)




def convergence_epoch(validation_epochs, validation_losses,
                      tolerance=0.05, patience=3):
    """
    Return the first actual epoch whose validation loss is within 5% of
    the best observed validation loss for `patience` consecutive checks.
    """
    epochs = np.asarray(validation_epochs, dtype=int)
    losses = np.asarray(validation_losses, dtype=float)
    if len(losses) < patience or not np.all(np.isfinite(losses)):
        return np.nan
    best = losses.min()
    threshold = best + tolerance * abs(best)
    for start in range(len(losses) - patience + 1):
        if np.all(losses[start:start + patience] <= threshold):
            return int(epochs[start])
    return np.nan


def train(
    seed,
    content_weight=1.0,
    style_weight=10.0,
    tv_weight=1e-5,
    lr=1e-3,
    num_epochs=DEFAULT_NUM_EPOCHS,
    validation_interval=VALIDATION_INTERVAL,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    min_relative_improvement=MIN_RELATIVE_IMPROVEMENT,
    checkpoint_dir=None,
):
    set_reproducible_seed(seed)

    content_nodes = ["relu_3_3"]
    style_nodes = ["relu_1_2", "relu_2_2", "relu_3_3", "relu_4_3"]
    return_nodes = {
        3: "relu_1_2",
        8: "relu_2_2",
        15: "relu_3_3",
        22: "relu_4_3",
    }

    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
    for parameter in vgg.parameters():
        parameter.requires_grad = False
    loss_network = create_feature_extractor(vgg, return_nodes).to(device).eval()

    model = StyleTransferNetwork(num_style=NUM_STYLE).to(device)
    optimizer = Adam(model.parameters(), lr=lr)

    fixed_style_features = {}
    with torch.no_grad():
        for style_id, image_path in STYLE_IMAGE_PATHS.items():
            image = process_single_image(image_path, 256, 240, device)
            fixed_style_features[style_id] = loss_network(image)

    epoch_records = []
    validation_records = []
    best_val_loss = float("inf")
    best_epoch = np.nan
    best_model_state = None
    checks_without_improvement = 0
    stopped_early = False
    cumulative_training_seconds = 0.0
    cumulative_validation_seconds = 0.0

    cuda_sync()
    run_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        cuda_sync()
        training_start = time.perf_counter()
        model.train()
        train_sums = {
            "content_loss": 0.0,
            "style_loss": 0.0,
            "tv_loss": 0.0,
            "weighted_content": 0.0,
            "weighted_style": 0.0,
            "weighted_tv": 0.0,
            "total_loss": 0.0,
        }
        train_images = 0

        for _, (content_images, _) in enumerate(content_dataloader):
            style_id = random.randint(0, NUM_STYLE - 1)
            content_images = content_images.to(device)
            current_batch_size = content_images.shape[0]
            style_codes = torch.zeros(
                current_batch_size, NUM_STYLE, 1, device=device
            )
            style_codes[:, style_id, :] = 1

            output_images = model(content_images, style_codes)
            output_features = loss_network(output_images)
            content_features = loss_network(content_images)
            content_loss = calc_content_loss(
                output_features, content_features, content_nodes
            )
            style_loss = calc_style_loss_custom(
                output_features, fixed_style_features[style_id], style_nodes
            )
            tv_loss = calc_tv_loss(output_images)
            weighted_content = content_weight * content_loss
            weighted_style = style_weight * style_loss
            weighted_tv = tv_weight * tv_loss
            total_loss = weighted_content + weighted_style + weighted_tv

            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            optimizer.step()

            values = {
                "content_loss": content_loss,
                "style_loss": style_loss,
                "tv_loss": tv_loss,
                "weighted_content": weighted_content,
                "weighted_style": weighted_style,
                "weighted_tv": weighted_tv,
                "total_loss": total_loss,
            }
            for name, value in values.items():
                train_sums[name] += value.item() * current_batch_size
            train_images += current_batch_size

        cuda_sync()
        training_seconds = time.perf_counter() - training_start
        cumulative_training_seconds += training_seconds

        # Validation is intentionally infrequent to conserve computation.
        should_validate = (
            epoch % validation_interval == 0 or epoch == num_epochs
        )
        val_means = {
            "content_loss": np.nan,
            "style_loss": np.nan,
            "tv_loss": np.nan,
            "total_loss": np.nan,
        }
        validation_seconds = 0.0

        if should_validate:
            cuda_sync()
            validation_start = time.perf_counter()
            model.eval()
            val_sums = {name: 0.0 for name in val_means}
            val_evaluations = 0

            with torch.no_grad():
                for val_content_images, _ in testing_dataloader:
                    val_content_images = val_content_images.to(device)
                    current_batch_size = val_content_images.shape[0]
                    content_features = loss_network(val_content_images)

                    for style_id in range(NUM_STYLE):
                        style_codes = torch.zeros(
                            current_batch_size, NUM_STYLE, 1, device=device
                        )
                        style_codes[:, style_id, :] = 1
                        output_images = model(val_content_images, style_codes)
                        output_features = loss_network(output_images)
                        content_loss = calc_content_loss(
                            output_features, content_features, content_nodes
                        )
                        style_loss = calc_style_loss_custom(
                            output_features,
                            fixed_style_features[style_id],
                            style_nodes,
                        )
                        tv_loss = calc_tv_loss(output_images)
                        total_loss = (
                            content_weight * content_loss
                            + style_weight * style_loss
                            + tv_weight * tv_loss
                        )
                        val_sums["content_loss"] += (
                            content_loss.item() * current_batch_size
                        )
                        val_sums["style_loss"] += (
                            style_loss.item() * current_batch_size
                        )
                        val_sums["tv_loss"] += tv_loss.item() * current_batch_size
                        val_sums["total_loss"] += (
                            total_loss.item() * current_batch_size
                        )
                        val_evaluations += current_batch_size

            cuda_sync()
            validation_seconds = time.perf_counter() - validation_start
            cumulative_validation_seconds += validation_seconds
            val_means = {
                name: value / val_evaluations
                for name, value in val_sums.items()
            }

            current_val_loss = val_means["total_loss"]
            required_improvement = (
                min_relative_improvement * abs(best_val_loss)
                if np.isfinite(best_val_loss) else 0.0
            )
            if current_val_loss < best_val_loss - required_improvement:
                best_val_loss = current_val_loss
                best_epoch = epoch
                best_model_state = copy.deepcopy(model.state_dict())
                checks_without_improvement = 0
                if checkpoint_dir:
                    checkpoint_path = Path(checkpoint_dir)
                    checkpoint_path.mkdir(parents=True, exist_ok=True)
                    torch.save(
                        best_model_state,
                        checkpoint_path / f"experiment_1_seed_{seed}_best.pth",
                    )
            else:
                checks_without_improvement += 1

        elapsed_seconds = time.perf_counter() - run_start
        record = {
            "seed": seed,
            "epoch": epoch,
            **{
                f"train_{name}": value / train_images
                for name, value in train_sums.items()
            },
            **{f"val_{name}": value for name, value in val_means.items()},
            "validated": should_validate,
            "training_seconds": training_seconds,
            "validation_seconds": validation_seconds,
            "elapsed_seconds": elapsed_seconds,
        }
        epoch_records.append(record)
        if should_validate:
            validation_records.append(record)

        validation_text = (
            f"{record['val_total_loss']:.6f}"
            if should_validate else "not checked"
        )
        print(
            f"Experiment 1 | seed {seed} | epoch {epoch}/{num_epochs} | "
            f"train {record['train_total_loss']:.6f} | "
            f"val {validation_text} | train time {training_seconds:.2f} sec"
        )

        if (
            should_validate
            and checks_without_improvement >= early_stopping_patience
        ):
            stopped_early = True
            print(
                f"Early stopping at epoch {epoch}; "
                f"best validation epoch was {best_epoch}."
            )
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    cuda_sync()
    total_wall_seconds = time.perf_counter() - run_start
    validation_epochs = [row["epoch"] for row in validation_records]
    validation_losses = [row["val_total_loss"] for row in validation_records]
    converged_epoch = convergence_epoch(
        validation_epochs,
        validation_losses,
        tolerance=CONVERGENCE_TOLERANCE,
        patience=CONVERGENCE_PATIENCE,
    )
    if np.isnan(converged_epoch):
        convergence_seconds = np.nan
    else:
        convergence_seconds = next(
            row["elapsed_seconds"]
            for row in validation_records
            if row["epoch"] == converged_epoch
        )

    last_validation = validation_records[-1]
    epochs_completed = len(epoch_records)
    run_record = {
        "seed": seed,
        "maximum_epochs": num_epochs,
        "epochs_completed": epochs_completed,
        "parameter_updates": epochs_completed * len(content_dataloader),
        "stopped_early": stopped_early,
        "final_train_loss": epoch_records[-1]["train_total_loss"],
        "final_val_loss": last_validation["val_total_loss"],
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "convergence_epoch": converged_epoch,
        "convergence_seconds": convergence_seconds,
        "training_only_seconds": cumulative_training_seconds,
        "validation_seconds": cumulative_validation_seconds,
        "total_wall_seconds": total_wall_seconds,
        "mean_training_seconds_per_epoch": (
            cumulative_training_seconds / epochs_completed
        ),
    }
    return model, pd.DataFrame(epoch_records), run_record


In [ ]:
path = "/content/drive/MyDrive/Movie_Project/Second_time_model/Experiment_1/Seed_22/experiment_1_seed_22_best.pth"
model = StyleTransferNetwork()
model.load_state_dict(torch.load(path))
model.eval()


StyleTransferNetwork(
  (conv1): ConvWithCIN(
    (padding): ReflectionPad2d((4, 4, 4, 4))
    (conv): Conv2d(3, 32, kernel_size=(9, 9), stride=(1, 1))
    (cin): CIN(
      (normalize): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    )
    (activation): ReLU()
  )
  (conv2): ConvWithCIN(
    (padding): ReflectionPad2d((1, 1, 1, 1))
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (cin): CIN(
      (normalize): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    )
    (activation): ReLU()
  )
  (conv3): ConvWithCIN(
    (padding): ReflectionPad2d((1, 1, 1, 1))
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2))
    (cin): CIN(
      (normalize): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    )
    (activation): ReLU()
  )
  (residual1): ResidualBlock(
    (conv1): ConvWithCIN(
      (padding): ReflectionPad2d((1, 1, 1, 1))
      (conv): Co

**Evaluation Statistic for A and B channel**

In [ ]:
STYLE_FOLDERS = {
    0: "Tim_Alice",
    1: "Tim_Beetlejuice",
    2: "We_Budapest",
    3: "We_Moonrise",
}
GENERATED_ROOT = Path("/content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1")

STYLE_ROOT = Path("/content/drive/MyDrive/Movie_Project/Directors_still_images")

EVALUATION_OUTPUT = Path("/content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1")

In [ ]:
# Experiment 1: baseline-relative improvement for A and B: A and B are calculated and reported separately.
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from PIL import Image
from scipy import stats
from scipy.stats import wasserstein_distance
from torchvision import transforms as T

SEEDS = [11, 22, 33]

STYLE_FOLDERS = {
    0: "Tim_Alice",
    1: "Tim_Beetlejuice",
    2: "We_Budapest",
    3: "We_Moonrise",
}

STYLE_ROOT = Path("/content/drive/MyDrive/Movie_Project/Directors_still_images")

GENERATED_ROOT = Path("/content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1")

EVALUATION_OUTPUT = GENERATED_ROOT / "Evaluation"
EVALUATION_OUTPUT.mkdir(parents=True, exist_ok=True)

evaluation_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(240),
])


def collect_image_paths(folder):
    supported_extensions = {".jpg", ".jpeg", ".png"}
    return sorted([
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in supported_extensions
    ])


def find_content_image(content_id):
    for suffix in (".jpg", ".jpeg", ".png"):
        path = Path(testing_dataset_path) / f"{content_id}{suffix}"
        return path


def student_t_summary(values, confidence=0.95):
    """Mean, sample SD, and two-sided Student t CI."""
    values = pd.Series(
        values,
        dtype=float,
    ).dropna().to_numpy()

    n = len(values)
    mean_value = float(np.mean(values))

    if n < 2:
        return {
            "n": n,
            "mean": mean_value,
            "sd": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    sd_value = float(np.std(values, ddof=1))
    critical_t = stats.t.ppf(
        (1 + confidence) / 2,
        df=n - 1,
    )
    margin = critical_t * sd_value / np.sqrt(n)

    return {
        "n": n,
        "mean": mean_value,
        "sd": sd_value,
        "ci_low": mean_value - margin,
        "ci_high": mean_value + margin,
    }

NUM_AB_BINS = 32

AB_CHANNEL_INDEX = {
    "A": 1,
    "B": 2,
}

# For float RGB input, OpenCV represents the CIELAB
# A and B channels approximately within this range.
AB_CHANNEL_RANGE = (-128.0, 127.0)

def pil_to_normalized_AB_histograms(
    image,
    num_bins=NUM_AB_BINS,
):
    """
    Convert a PIL RGB image into separate normalized
    32-bin CIELAB A- and B-channel histograms.
    """
    rgb = np.asarray(
        image.convert("RGB"),
        dtype=np.float32,
    ) / 255.0

    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB,
    )

    histograms = {}

    for channel_name, channel_index in (
        AB_CHANNEL_INDEX.items()
    ):
        counts, bin_edges = np.histogram(
            lab[:, :, channel_index],
            bins=num_bins,
            range=AB_CHANNEL_RANGE,
        )

        histogram = counts.astype(np.float64)

        if histogram.sum() == 0:
            raise ValueError(
                f"{channel_name}-channel histogram "
                "contains no pixels."
            )

        histogram /= histogram.sum()

        bin_centers = (
            bin_edges[:-1] + bin_edges[1:]
        ) / 2.0

        histograms[channel_name] = {
            "histogram": histogram,
            "bin_centers": bin_centers,
        }

    return histograms


def AB_histogram_distance(
    first,
    second,
):
    """Calculate Wasserstein-1 distance between two histograms."""
    return wasserstein_distance(
        first["bin_centers"],
        second["bin_centers"],
        u_weights=first["histogram"],
        v_weights=second["histogram"],
    )


# ---------------------------------------------------------
# 1. Precompute the reference A and B histograms.
# ---------------------------------------------------------
AB_reference_histograms = {}

for style_id, style_name in STYLE_FOLDERS.items():
    style_folder = STYLE_ROOT / style_name
    reference_paths = collect_image_paths(
        style_folder
    )

    if len(reference_paths) != 14:
        raise ValueError(
            f"{style_name} contains "
            f"{len(reference_paths)} frames; expected 14."
        )

    AB_reference_histograms[style_id] = []

    for reference_path in reference_paths:
        reference_image = Image.open(
            reference_path
        ).convert("RGB")

        reference_image = evaluation_transform(
            reference_image
        )

        AB_reference_histograms[
            style_id
        ].append({
            "frame_name": reference_path.name,
            "histograms":
                pil_to_normalized_AB_histograms(
                    reference_image
                ),
        })

print("Reference A and B histograms prepared.")


# ---------------------------------------------------------
# 2. Calculate the content baseline once.
# ---------------------------------------------------------
AB_baseline_rows = []

for content_id in range(1, 81):
    content_path = find_content_image(
        content_id
    )

    content_image = Image.open(
        content_path
    ).convert("RGB")

    content_image = evaluation_transform(
        content_image
    )

    content_histograms = (
        pil_to_normalized_AB_histograms(
            content_image
        )
    )

    for style_id, style_name in (
        STYLE_FOLDERS.items()
    ):
        row = {
            "content_id": content_id,
            "style_id": style_id,
            "style_name": style_name,
            "n_reference_frames": 14,
        }

        for channel_name in ("A", "B"):
            frame_distances = [
                AB_histogram_distance(
                    content_histograms[
                        channel_name
                    ],
                    reference["histograms"][
                        channel_name
                    ],
                )
                for reference in
                AB_reference_histograms[
                    style_id
                ]
            ]

            row[
                f"content_baseline_"
                f"{channel_name}_distance"
            ] = float(
                np.mean(frame_distances)
            )

            row[
                f"sd_content_{channel_name}_"
                "distance_across_frames"
            ] = float(
                np.std(
                    frame_distances,
                    ddof=1,
                )
            )

        AB_baseline_rows.append(row)

AB_content_baseline_df = pd.DataFrame(
    AB_baseline_rows
)

assert len(AB_content_baseline_df) == 80 * 4
assert not AB_content_baseline_df.duplicated(
    ["content_id", "style_id"]
).any()

print(
    "A/B content-baseline rows:",
    len(AB_content_baseline_df),
)


# ---------------------------------------------------------
# 3. Calculate every generated output's A and B distances.
# ---------------------------------------------------------
AB_output_rows = []

for seed in SEEDS:
    seed_folder = (
        GENERATED_ROOT / f"Seed_{seed}"
    )

    if not seed_folder.exists():
        raise FileNotFoundError(
            f"Missing generated-output folder: "
            f"{seed_folder}"
        )

    for content_id in range(1, 81):
        for style_id, style_name in (
            STYLE_FOLDERS.items()
        ):
            generated_path = (
                seed_folder
                / f"Image_{content_id}_"
                  f"style_{style_id}.jpg"
            )

            if not generated_path.exists():
                raise FileNotFoundError(
                    f"Missing generated image: "
                    f"{generated_path}"
                )

            generated_image = Image.open(
                generated_path
            ).convert("RGB")

            if generated_image.size != (
                240,
                240,
            ):
                raise ValueError(
                    f"{generated_path} has size "
                    f"{generated_image.size}; "
                    "expected (240, 240)."
                )

            generated_histograms = (
                pil_to_normalized_AB_histograms(
                    generated_image
                )
            )

            row = {
                "experiment": 1,
                "seed": seed,
                "content_id": content_id,
                "style_id": style_id,
                "style_name": style_name,
                "n_reference_frames": 14,
            }

            for channel_name in ("A", "B"):
                frame_distances = [
                    AB_histogram_distance(
                        generated_histograms[
                            channel_name
                        ],
                        reference[
                            "histograms"
                        ][channel_name],
                    )
                    for reference in
                    AB_reference_histograms[
                        style_id
                    ]
                ]

                row[
                    f"mean_{channel_name}_distance"
                ] = float(
                    np.mean(frame_distances)
                )

                row[
                    f"sd_{channel_name}_"
                    "distance_across_frames"
                ] = float(
                    np.std(
                        frame_distances,
                        ddof=1,
                    )
                )

            AB_output_rows.append(row)

AB_output_distance_df = pd.DataFrame(
    AB_output_rows
)

assert len(AB_output_distance_df) == (
    3 * 80 * 4
)

assert not AB_output_distance_df.duplicated(
    ["seed", "content_id", "style_id"]
).any()

print(
    "A/B generated-output rows:",
    len(AB_output_distance_df),
)


# ---------------------------------------------------------
# 4. Calculate baseline-relative A and B improvement.
# ---------------------------------------------------------
AB_improvement_df = (
    AB_output_distance_df.merge(
        AB_content_baseline_df,
        on=[
            "content_id",
            "style_id",
            "style_name",
            "n_reference_frames",
        ],
        how="left",
        validate="many_to_one",
    )
)

for channel_name in ("A", "B"):
    AB_improvement_df[
        f"{channel_name}_improvement"
    ] = (
        AB_improvement_df[
            f"content_baseline_"
            f"{channel_name}_distance"
        ]
        -
        AB_improvement_df[
            f"mean_{channel_name}_distance"
        ]
    )

    AB_improvement_df[
        f"{channel_name}_"
        "distance_improved"
    ] = (
        AB_improvement_df[
            f"{channel_name}_improvement"
        ] > 0
    ).astype(int)

assert len(AB_improvement_df) == (
    3 * 80 * 4
)

assert not AB_improvement_df.isna().any().any()


# ---------------------------------------------------------
# 5. Summarize the 80 images within each seed.
# ---------------------------------------------------------
AB_seed_summary_rows = []

for (
    seed,
    style_id,
    style_name,
), group in AB_improvement_df.groupby([
    "seed",
    "style_id",
    "style_name",
]):
    row = {
        "experiment": 1,
        "seed": seed,
        "style_id": style_id,
        "style_name": style_name,
        "n_content_images": len(group),
    }

    for channel_name in ("A", "B"):
        row[
            f"mean_content_baseline_"
            f"{channel_name}_distance"
        ] = group[
            f"content_baseline_"
            f"{channel_name}_distance"
        ].mean()

        row[
            f"mean_output_"
            f"{channel_name}_distance"
        ] = group[
            f"mean_{channel_name}_distance"
        ].mean()

        row[
            f"mean_{channel_name}_"
            "improvement"
        ] = group[
            f"{channel_name}_improvement"
        ].mean()

        row[
            f"sd_{channel_name}_"
            "improvement_across_images"
        ] = group[
            f"{channel_name}_improvement"
        ].std(ddof=1)

        row[
            f"percentage_images_"
            f"{channel_name}_improved"
        ] = (
            group[
                f"{channel_name}_"
                "distance_improved"
            ].mean()
            * 100
        )

    AB_seed_summary_rows.append(row)

AB_improvement_by_seed_df = pd.DataFrame(
    AB_seed_summary_rows
)


# ---------------------------------------------------------
# 6. Publication summary across the three seeds.
#    The 95% CI is reported only for improvement.
# ---------------------------------------------------------
AB_publication_rows = []

for style_id, style_name in (
    STYLE_FOLDERS.items()
):
    baseline_style = (
        AB_content_baseline_df.loc[
            AB_content_baseline_df[
                "style_id"
            ] == style_id
        ]
    )

    seed_style = (
        AB_improvement_by_seed_df.loc[
            AB_improvement_by_seed_df[
                "style_id"
            ] == style_id
        ]
    )

    row = {
        "style_id": style_id,
        "style_name": style_name,
        "n_testing_images": 80,
        "n_seeds": 3,
    }

    for channel_name in ("A", "B"):
        baseline_values = baseline_style[
            f"content_baseline_"
            f"{channel_name}_distance"
        ]

        output_values = seed_style[
            f"mean_output_"
            f"{channel_name}_distance"
        ]

        improvement_values = seed_style[
            f"mean_{channel_name}_"
            "improvement"
        ]

        improvement_stats = (
            student_t_summary(
                improvement_values
            )
        )

        row[
            f"testing_baseline_mean_"
            f"{channel_name}_distance"
        ] = baseline_values.mean()

        row[
            f"testing_baseline_sd_"
            f"{channel_name}_across_images"
        ] = baseline_values.std(ddof=1)

        row[
            f"stylized_output_mean_"
            f"{channel_name}_distance"
        ] = output_values.mean()

        row[
            f"stylized_output_sd_"
            f"{channel_name}_across_seeds"
        ] = output_values.std(ddof=1)

        row[
            f"baseline_relative_mean_"
            f"{channel_name}_improvement"
        ] = improvement_stats["mean"]

        row[
            f"{channel_name}_improvement_"
            "sd_across_seeds"
        ] = improvement_stats["sd"]

        row[
            f"{channel_name}_improvement_"
            "95_ci_low"
        ] = improvement_stats["ci_low"]

        row[
            f"{channel_name}_improvement_"
            "95_ci_high"
        ] = improvement_stats["ci_high"]

        row[
            f"mean_percentage_images_"
            f"{channel_name}_improved"
        ] = seed_style[
            f"percentage_images_"
            f"{channel_name}_improved"
        ].mean()

    AB_publication_rows.append(row)

AB_publication_summary_df = pd.DataFrame(
    AB_publication_rows
)


# ---------------------------------------------------------
# 7. Save A- and B-channel evaluation tables.
# ---------------------------------------------------------
AB_content_baseline_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_AB_content_baseline.csv",
    index=False,
)

AB_output_distance_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_AB_distances_per_image.csv",
    index=False,
)

AB_improvement_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_AB_content_improvement_per_output.csv",
    index=False,
)

AB_improvement_by_seed_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_AB_content_improvement_by_seed.csv",
    index=False,
)

AB_publication_summary_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_AB_publication_summary_by_style.csv",
    index=False,
)

print(
    "Saved A/B evaluation tables to:",
    EVALUATION_OUTPUT,
)

display(
    AB_improvement_by_seed_df.round(3)
)

display(
    AB_publication_summary_df.round(3)
)


Reference A and B histograms prepared.
A/B content-baseline rows: 320
A/B generated-output rows: 960
Saved A/B evaluation tables to: /content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1/Evaluation


,experiment,seed,style_id,style_name,n_content_images,mean_content_baseline_A_distance,mean_output_A_distance,mean_A_improvement,sd_A_improvement_across_images,percentage_images_A_improved,mean_content_baseline_B_distance,mean_output_B_distance,mean_B_improvement,sd_B_improvement_across_images,percentage_images_B_improved
0,1,11,0,Tim_Alice,80,7.664,5.812,1.852,5.158,43.75,13.056,14.420,-1.365,9.294,28.75
1,1,11,1,Tim_Beetlejuice,80,11.353,13.837,-2.484,4.880,16.25,18.818,16.052,2.766,8.365,43.75
2,1,11,2,We_Budapest,80,22.917,6.371,16.546,6.873,100.00,14.924,10.184,4.740,6.769,73.75
3,1,11,3,We_Moonrise,80,7.898,5.602,2.296,4.997,58.75,20.299,6.434,13.864,7.488,100.00
4,1,22,0,Tim_Alice,80,7.664,5.830,1.834,5.063,45.00,13.056,16.014,-2.959,9.337,25.00
5,1,22,1,Tim_Beetlejuice,80,11.353,14.516,-3.164,4.998,15.00,18.818,15.237,3.581,8.335,48.75
6,1,22,2,We_Budapest,80,22.917,6.116,16.801,6.904,100.00,14.924,10.371,4.553,6.754,72.50
7,1,22,3,We_Moonrise,80,7.898,7.016,0.881,4.990,32.50,20.299,6.482,13.816,7.477,100.00
8,1,33,0,Tim_Alice,80,7.664,5.741,1.923,5.096,47.50,13.056,13.590,-0.534,9.297,31.25
9,1,33,1,Tim_Beetlejuice,80,11.353,14.147,-2.794,4.941,16.25,18.818,15.110,3.708,8.278,50.00


,style_id,style_name,n_testing_images,n_seeds,testing_baseline_mean_A_distance,testing_baseline_sd_A_across_images,stylized_output_mean_A_distance,stylized_output_sd_A_across_seeds,baseline_relative_mean_A_improvement,A_improvement_sd_across_seeds,...,mean_percentage_images_A_improved,testing_baseline_mean_B_distance,testing_baseline_sd_B_across_images,stylized_output_mean_B_distance,stylized_output_sd_B_across_seeds,baseline_relative_mean_B_improvement,B_improvement_sd_across_seeds,B_improvement_95_ci_low,B_improvement_95_ci_high,mean_percentage_images_B_improved
0,0,Tim_Alice,80,3,7.664,4.999,5.795,0.047,1.870,0.047,...,45.417,13.056,9.319,14.675,1.232,-1.619,1.232,-4.680,1.442,28.333
1,1,Tim_Beetlejuice,80,3,11.353,4.903,14.167,0.340,-2.814,0.340,...,15.833,18.818,8.319,15.466,0.511,3.352,0.511,2.082,4.621,47.500
2,2,We_Budapest,80,3,22.917,6.895,6.120,0.249,16.797,0.249,...,100.000,14.924,6.750,10.381,0.203,4.543,0.203,4.039,5.046,72.083
3,3,We_Moonrise,80,3,7.898,4.998,6.536,0.809,1.362,0.809,...,40.833,20.299,7.490,6.489,0.059,13.809,0.059,13.664,13.955,100.000


# **L-channel**

In [ ]:
# ---------------------------------------------------------
# Experiment 1: content-image baseline for L distributions
# ---------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

from PIL import Image
from scipy.stats import wasserstein_distance
from torchvision import transforms as T

SEEDS = [11, 22, 33]
NUM_L_BINS = 32

STYLE_FOLDERS = {
    0: "Tim_Alice",
    1: "Tim_Beetlejuice",
    2: "We_Budapest",
    3: "We_Moonrise",
}

STYLE_ROOT = Path(
    "/content/drive/MyDrive/Movie_Project/Directors_still_images"
)

GENERATED_ROOT = Path(
    "/content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1"
)

EVALUATION_OUTPUT = GENERATED_ROOT / "Evaluation"
EVALUATION_OUTPUT.mkdir(parents=True, exist_ok=True)

# This matches the deterministic preprocessing used when
# generating the stylized testing images.
evaluation_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(240),
])


def collect_image_paths(folder):
    supported_extensions = {".jpg", ".jpeg", ".png"}
    return sorted([
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in supported_extensions
    ])


def find_content_image(content_id):
    for suffix in (".jpg", ".jpeg", ".png"):
        path = Path(testing_dataset_path) / f"{content_id}{suffix}"
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Missing testing content image {content_id}."
    )


def pil_to_normalized_L_histogram(image, num_bins=NUM_L_BINS):
    """
    Convert a PIL RGB image into a normalized
    32-bin CIELAB L-channel histogram.
    """
    rgb = np.asarray(
        image.convert("RGB"),
        dtype=np.float32,
    ) / 255.0

    # For float RGB input in [0, 1], OpenCV returns
    # L values in approximately [0, 100].
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L_channel = lab[:, :, 0]

    counts, bin_edges = np.histogram(
        L_channel,
        bins=num_bins,
        range=(0.0, 100.0),
    )

    histogram = counts.astype(np.float64)

    if histogram.sum() == 0:
        raise ValueError(
            "L-channel histogram contains no pixels."
        )

    histogram /= histogram.sum()

    bin_centers = (
        bin_edges[:-1] + bin_edges[1:]
    ) / 2.0

    return histogram, bin_centers


def L_histogram_distance(
    first_histogram,
    second_histogram,
    bin_centers,
):
    """Calculate Wasserstein-1 distance between two L histograms."""
    return wasserstein_distance(
        bin_centers,
        bin_centers,
        u_weights=first_histogram,
        v_weights=second_histogram,
    )


# ---------------------------------------------------------
# 1. Precompute the 14 reference L histograms per style.
# ---------------------------------------------------------
reference_histograms = {}

for style_id, style_name in STYLE_FOLDERS.items():
    style_folder = STYLE_ROOT / style_name
    reference_paths = collect_image_paths(style_folder)

    if len(reference_paths) != 14:
        raise ValueError(
            f"{style_name} contains {len(reference_paths)} "
            "frames; expected 14."
        )

    reference_histograms[style_id] = []

    for reference_path in reference_paths:
        reference_image = Image.open(
            reference_path
        ).convert("RGB")

        reference_image = evaluation_transform(
            reference_image
        )

        histogram, bin_centers = (
            pil_to_normalized_L_histogram(
                reference_image
            )
        )

        reference_histograms[style_id].append({
            "frame_name": reference_path.name,
            "histogram": histogram,
        })

print("Reference L histograms prepared for all four styles.")


# ---------------------------------------------------------
# 2. Calculate the testing-content baseline once.
# ---------------------------------------------------------
content_baseline_rows = []

for content_id in range(1, 81):
    content_path = find_content_image(content_id)

    content_image = Image.open(
        content_path
    ).convert("RGB")

    processed_content_image = evaluation_transform(
        content_image
    )

    content_histogram, bin_centers = (
        pil_to_normalized_L_histogram(
            processed_content_image
        )
    )

    for style_id, style_name in STYLE_FOLDERS.items():
        frame_distances = []

        for reference in reference_histograms[style_id]:
            distance = L_histogram_distance(
                content_histogram,
                reference["histogram"],
                bin_centers,
            )
            frame_distances.append(distance)

        content_baseline_rows.append({
            "content_id": content_id,
            "style_id": style_id,
            "style_name": style_name,
            "n_reference_frames": len(frame_distances),
            "content_baseline_L_distance": float(
                np.mean(frame_distances)
            ),
            "sd_content_L_distance_across_frames": float(
                np.std(frame_distances, ddof=1)
            ),
        })

content_baseline_df = pd.DataFrame(
    content_baseline_rows
)

assert len(content_baseline_df) == 80 * 4
assert (
    content_baseline_df["n_reference_frames"] == 14
).all()
assert not content_baseline_df.duplicated(
    ["content_id", "style_id"]
).any()
assert not content_baseline_df[
    "content_baseline_L_distance"
].isna().any()

print("Content-baseline rows:", len(content_baseline_df))


# ---------------------------------------------------------
# 3. Calculate every generated output's L distance.
# ---------------------------------------------------------
output_rows = []

for seed in SEEDS:
    seed_folder = GENERATED_ROOT / f"Seed_{seed}"

    if not seed_folder.exists():
        raise FileNotFoundError(
            f"Missing generated-output folder: {seed_folder}. "
            "Save each seed's outputs in its own Seed_11, "
            "Seed_22, or Seed_33 folder."
        )

    for content_id in range(1, 81):
        for style_id, style_name in STYLE_FOLDERS.items():
            generated_path = seed_folder / (
                f"Image_{content_id}_style_{style_id}.jpg"
            )

            if not generated_path.exists():
                raise FileNotFoundError(
                    f"Missing generated image: {generated_path}"
                )

            generated_image = Image.open(
                generated_path
            ).convert("RGB")

            # Stylized outputs already inherit the 240 x 240
            # dimensions of the preprocessed testing inputs.
            if generated_image.size != (240, 240):
                raise ValueError(
                    f"{generated_path} has size "
                    f"{generated_image.size}; expected (240, 240)."
                )

            generated_histogram, bin_centers = (
                pil_to_normalized_L_histogram(
                    generated_image
                )
            )

            frame_distances = []

            for reference in reference_histograms[style_id]:
                distance = L_histogram_distance(
                    generated_histogram,
                    reference["histogram"],
                    bin_centers,
                )
                frame_distances.append(distance)

            output_rows.append({
                "experiment": 1,
                "seed": seed,
                "content_id": content_id,
                "style_id": style_id,
                "style_name": style_name,
                "n_reference_frames": len(frame_distances),
                "mean_L_distance": float(
                    np.mean(frame_distances)
                ),
                "sd_L_distance_across_frames": float(
                    np.std(frame_distances, ddof=1)
                ),
            })

output_distance_df = pd.DataFrame(output_rows)

assert len(output_distance_df) == 3 * 80 * 4
assert (
    output_distance_df["n_reference_frames"] == 14
).all()
assert not output_distance_df.duplicated(
    ["seed", "content_id", "style_id"]
).any()
assert not output_distance_df[
    "mean_L_distance"
].isna().any()

print("Generated-output rows:", len(output_distance_df))


# ---------------------------------------------------------
# 4. Baseline-relative L improvement
#    Improvement = baseline distance - output distance
# ---------------------------------------------------------
improvement_df = output_distance_df.merge(
    content_baseline_df[[
        "content_id",
        "style_id",
        "style_name",
        "content_baseline_L_distance",
    ]],
    on=[
        "content_id",
        "style_id",
        "style_name",
    ],
    how="left",
    validate="many_to_one",
)

improvement_df["L_improvement"] = (
    improvement_df["content_baseline_L_distance"]
    - improvement_df["mean_L_distance"]
)

improvement_df["lighting_distance_improved"] = (
    improvement_df["L_improvement"] > 0
).astype(int)

assert len(improvement_df) == 3 * 80 * 4
assert not improvement_df[
    "content_baseline_L_distance"
].isna().any()


# ---------------------------------------------------------
# 5. Summarize the 80 testing images within each seed.
# ---------------------------------------------------------
improvement_by_seed = (
    improvement_df
    .groupby([
        "seed",
        "style_id",
        "style_name",
    ])
    .agg(
        n_content_images=(
            "content_id",
            "count",
        ),
        mean_content_baseline_L_distance=(
            "content_baseline_L_distance",
            "mean",
        ),
        mean_output_L_distance=(
            "mean_L_distance",
            "mean",
        ),
        mean_L_improvement=(
            "L_improvement",
            "mean",
        ),
        sd_L_improvement_across_images=(
            "L_improvement",
            "std",
        ),
        median_L_improvement=(
            "L_improvement",
            "median",
        ),
        n_images_improved=(
            "lighting_distance_improved",
            "sum",
        ),
        proportion_images_improved=(
            "lighting_distance_improved",
            "mean",
        ),
    )
    .reset_index()
)

improvement_by_seed[
    "percentage_images_improved"
] = (
    improvement_by_seed[
        "proportion_images_improved"
    ] * 100
)


# ---------------------------------------------------------
# 6. Summarize the three seed-level results by style.
# ---------------------------------------------------------
baseline_by_style = (
    content_baseline_df
    .groupby([
        "style_id",
        "style_name",
    ])
    .agg(
        mean_content_baseline_L_distance=(
            "content_baseline_L_distance",
            "mean",
        ),
        sd_content_baseline_L_distance_across_images=(
            "content_baseline_L_distance",
            "std",
        ),
    )
    .reset_index()
)

improvement_across_seeds = (
    improvement_by_seed
    .groupby([
        "style_id",
        "style_name",
    ])
    .agg(
        mean_output_L_distance=(
            "mean_output_L_distance",
            "mean",
        ),
        sd_output_L_distance_across_seeds=(
            "mean_output_L_distance",
            "std",
        ),
        mean_L_improvement=(
            "mean_L_improvement",
            "mean",
        ),
        sd_L_improvement_across_seeds=(
            "mean_L_improvement",
            "std",
        ),
        mean_percentage_images_improved=(
            "percentage_images_improved",
            "mean",
        ),
        sd_percentage_images_improved_across_seeds=(
            "percentage_images_improved",
            "std",
        ),
        minimum_percentage_images_improved=(
            "percentage_images_improved",
            "min",
        ),
        maximum_percentage_images_improved=(
            "percentage_images_improved",
            "max",
        ),
    )
    .reset_index()
)

improvement_across_seeds = (
    baseline_by_style.merge(
        improvement_across_seeds,
        on=[
            "style_id",
            "style_name",
        ],
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# 7. Save all L-channel evaluation tables.
# ---------------------------------------------------------
content_baseline_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_L_content_baseline.csv",
    index=False,
)

output_distance_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_L_distances_per_image.csv",
    index=False,
)

improvement_df.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_L_content_improvement_per_output.csv",
    index=False,
)

improvement_by_seed.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_L_content_improvement_by_seed.csv",
    index=False,
)

improvement_across_seeds.to_csv(
    EVALUATION_OUTPUT
    / "experiment_1_L_content_improvement_across_seeds.csv",
    index=False,
)

print("Saved L-channel evaluation tables to:", EVALUATION_OUTPUT)
display(improvement_by_seed)
display(improvement_across_seeds)


Reference L histograms prepared for all four styles.
Content-baseline rows: 320
Generated-output rows: 960
Saved L-channel evaluation tables to: /content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1/Evaluation


,seed,style_id,style_name,n_content_images,mean_content_baseline_L_distance,mean_output_L_distance,mean_L_improvement,sd_L_improvement_across_images,median_L_improvement,n_images_improved,proportion_images_improved,percentage_images_improved
0,11,0,Tim_Alice,80,33.502001,21.651693,11.850308,10.708521,12.509479,68,0.8500,85.00
1,11,1,Tim_Beetlejuice,80,36.290158,28.664074,7.626084,10.857337,8.221118,62,0.7750,77.50
2,11,2,We_Budapest,80,17.265626,12.765265,4.500362,6.220074,3.344448,56,0.7000,70.00
3,11,3,We_Moonrise,80,20.164668,16.283671,3.880997,5.823644,2.498217,57,0.7125,71.25
4,22,0,Tim_Alice,80,33.502001,18.594966,14.907035,10.677334,15.640098,73,0.9125,91.25
5,22,1,Tim_Beetlejuice,80,36.290158,32.391281,3.898878,10.792310,4.533622,51,0.6375,63.75
6,22,2,We_Budapest,80,17.265626,12.727643,4.537984,6.241617,3.388315,57,0.7125,71.25
7,22,3,We_Moonrise,80,20.164668,14.928376,5.236292,5.806860,3.851469,68,0.8500,85.00
8,33,0,Tim_Alice,80,33.502001,15.892378,17.609623,10.656178,18.237979,76,0.9500,95.00
9,33,1,Tim_Beetlejuice,80,36.290158,33.122593,3.167565,10.829163,3.655603,50,0.6250,62.50


,style_id,style_name,mean_content_baseline_L_distance,sd_content_baseline_L_distance_across_images,mean_output_L_distance,sd_output_L_distance_across_seeds,mean_L_improvement,sd_L_improvement_across_seeds,mean_percentage_images_improved,sd_percentage_images_improved_across_seeds,minimum_percentage_images_improved,maximum_percentage_images_improved
0,0,Tim_Alice,33.502001,10.810501,18.713012,2.881472,14.788989,2.881472,90.416667,5.051815,85.00,95.00
1,1,Tim_Beetlejuice,36.290158,10.910354,31.392649,2.391139,4.897509,2.391139,67.916667,8.322910,62.50,77.50
2,2,We_Budapest,17.265626,6.224764,12.865121,0.206397,4.400505,0.206397,69.583333,1.909407,67.50,71.25
3,3,We_Moonrise,20.164668,5.804602,15.476420,0.713861,4.688248,0.713861,79.583333,7.324332,71.25,85.00


In [ ]:
# ---------------------------------------------------------
# Publication summary: baseline, output, improvement, and 95% CI
# ---------------------------------------------------------
from scipy import stats


def student_t_summary(values, confidence=0.95):
    """
    Return the mean, sample SD, and two-sided Student's t
    confidence interval for a one-dimensional collection.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna().to_numpy()

    n = len(values)
    mean_value = float(np.mean(values))

    if n < 2:
        return {
            "n": n,
            "mean": mean_value,
            "sd": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    sd_value = float(
        np.std(values, ddof=1)
    )

    critical_t = stats.t.ppf(
        (1 + confidence) / 2,
        df=n - 1,
    )

    margin = (
        critical_t
        * sd_value
        / np.sqrt(n)
    )

    return {
        "n": n,
        "mean": mean_value,
        "sd": sd_value,
        "ci_low": mean_value - margin,
        "ci_high": mean_value + margin,
    }


publication_summary_rows = []

for style_id, style_name in STYLE_FOLDERS.items():
    # The content baseline is calculated once from
    # the 80 unchanged testing content images.
    baseline_values = content_baseline_df.loc[
        content_baseline_df["style_id"] == style_id,
        "content_baseline_L_distance",
    ]

    baseline_stats = student_t_summary(
        baseline_values
    )

    # Output and improvement uncertainty is summarized
    # across the three independently trained seed-level means.
    style_seed_results = improvement_by_seed.loc[
        improvement_by_seed["style_id"] == style_id
    ]

    output_stats = student_t_summary(
        style_seed_results[
            "mean_output_L_distance"
        ]
    )

    improvement_stats = student_t_summary(
        style_seed_results[
            "mean_L_improvement"
        ]
    )

    publication_summary_rows.append({
        "style_id": style_id,
        "style_name": style_name,

        "testing_baseline_n_images":
            baseline_stats["n"],
        "testing_baseline_mean_L_distance":
            baseline_stats["mean"],
        "testing_baseline_sd_across_images":
            baseline_stats["sd"],
        "testing_baseline_95_ci_low":
            baseline_stats["ci_low"],
        "testing_baseline_95_ci_high":
            baseline_stats["ci_high"],

        "stylized_output_n_seeds":
            output_stats["n"],
        "stylized_output_mean_L_distance":
            output_stats["mean"],
        "stylized_output_sd_across_seeds":
            output_stats["sd"],
        "stylized_output_95_ci_low":
            output_stats["ci_low"],
        "stylized_output_95_ci_high":
            output_stats["ci_high"],

        "baseline_relative_mean_L_improvement":
            improvement_stats["mean"],
        "improvement_sd_across_seeds":
            improvement_stats["sd"],
        "improvement_95_ci_low":
            improvement_stats["ci_low"],
        "improvement_95_ci_high":
            improvement_stats["ci_high"],

        "mean_percentage_images_improved":
            style_seed_results[
                "percentage_images_improved"
            ].mean(),
    })

publication_summary_df = pd.DataFrame(
    publication_summary_rows
)

publication_summary_path = (
    EVALUATION_OUTPUT
    / "experiment_1_L_publication_summary_by_style.csv"
)

publication_summary_df.to_csv(
    publication_summary_path,
    index=False,
)

print("Saved:", publication_summary_path)
display(publication_summary_df.round(3))


Saved: /content/drive/MyDrive/Movie_Project/Second_time_results/Experiment_1/Evaluation/experiment_1_L_publication_summary_by_style.csv


,style_id,style_name,testing_baseline_n_images,testing_baseline_mean_L_distance,testing_baseline_sd_across_images,testing_baseline_95_ci_low,testing_baseline_95_ci_high,stylized_output_n_seeds,stylized_output_mean_L_distance,stylized_output_sd_across_seeds,stylized_output_95_ci_low,stylized_output_95_ci_high,baseline_relative_mean_L_improvement,improvement_sd_across_seeds,improvement_95_ci_low,improvement_95_ci_high,mean_percentage_images_improved
0,0,Tim_Alice,80,33.502,10.811,31.096,35.908,3,18.713,2.881,11.555,25.871,14.789,2.881,7.631,21.947,90.417
1,1,Tim_Beetlejuice,80,36.290,10.910,33.862,38.718,3,31.393,2.391,25.453,37.333,4.898,2.391,-1.042,10.837,67.917
2,2,We_Budapest,80,17.266,6.225,15.880,18.651,3,12.865,0.206,12.352,13.378,4.401,0.206,3.888,4.913,69.583
3,3,We_Moonrise,80,20.165,5.805,18.873,21.456,3,15.476,0.714,13.703,17.250,4.688,0.714,2.915,6.462,79.583
